# 🧭 30-Day Spark Roadmap (Databricks-focused)

👉 **Environment:** Databricks Community Edition  
👉 **Storage layer later:** Delta Lake

---

## 🟢 WEEK 1 — Foundations (Days 1–7)

👉 **Goal:** Understand Spark + DataFrames (core mental model)

**Day 1–2 — Introduction to Apache Spark**
- What Spark is (distributed compute)
- Driver vs Executors
- Lazy evaluation

**Day 3 — Spark Runtime Architecture**
- Jobs, stages, tasks
- Basic execution flow

**Day 4 — Spark in Databricks**
- Notebooks
- Clusters (conceptually)
- Running your first job

**Day 5–6 — DataFrames & SQL**
- From curriculum: Introduction to Spark DataFrames and SQL
- Hands-on:
  - Load CSV
  - `select`, `filter`, `withColumn`
  - Basic SQL queries

**Day 7 — Reading & Writing Data**
- CSV / JSON / Parquet
- Write outputs

💡 **Mini-task:** Read raw data → write cleaned version

---

## 🟡 WEEK 2 — Core Transformations (Days 8–14)

👉 **Goal:** Translate ETL logic into Spark

**Day 8 — Distributed Programming Fundamentals**
- Transformations vs actions
- Immutability

**Day 9–10 — Basic ETL with DataFrame API**
- From curriculum: Basic ETL with the DataFrame API — Flight Data ETL
- Hands-on:
  - Clean messy dataset
  - Cast types
  - Handle nulls

**Day 11–12 — Grouping & Aggregations**
- From curriculum: Grouping and Aggregating Data
- Hands-on:
  - `groupBy`, `agg`
  - Multiple aggregations
  - Sorting results

💡 **Lab idea:** Revenue by country / Avg transactions per user

**Day 13 — Relational Operations**
- Joins (inner, left, right)
- Hands-on: Join customers + transactions

**Day 14 — Complex Data Types**
- Arrays, structs
- Hands-on: Explode arrays, work with nested JSON

---

## 🔵 WEEK 3 — Real Data + Streaming (Days 15–21)

👉 **Goal:** Work like a real data engineer

**Day 15–16 — Analyze Transaction Data**
- From curriculum: Analyzing Transaction Data with DataFrames
- Build: KPIs (total revenue, top users, trends)

**Day 17 — Mini Project (Batch Pipeline)**
- End-to-end: Read raw → Clean → Join → Aggregate → Write output

**Day 18 — Intro to Streaming**
- From curriculum: Introduction to Stream Processing
- Concepts: Batch vs streaming

**Day 19–20 — Structured Streaming**
- From curriculum: Spark Structured Streaming
- Hands-on:
  - Read streaming data (rate source or files)
  - Simple aggregation

**Day 21 — Window Aggregations (Streaming)**
- Sliding windows
- Time-based aggregations

💡 **Example:** Events per minute

---

## 🔴 WEEK 4 — Optimization + Delta (Days 22–30)

👉 **Goal:** Move from "it works" → "it's production-ready"

**Day 22 — Spark + Databricks Deep Dive**
- Execution plan (`explain()`)
- DAG understanding

**Day 23–24 — Delta Lake**
- From curriculum: Using Apache Spark with Delta Lake
- Hands-on:
  - Save as Delta
  - Update / merge
  - Time travel (optional)

**Day 25 — Optimization Basics**
- From curriculum: Optimizing Apache Spark
- Learn: Partitioning, Caching, Shuffle basics

**Day 26 — Optimization Lab**
- Compare: With vs without cache, different join strategies

**Day 27–28 — Final Project**
- Combine everything: Batch + streaming (optional), Delta output, Aggregations, Clean pipeline
- Example: E-commerce pipeline — ingest → clean → join → KPIs

**Day 29 — Review Weak Areas**
- Joins, Aggregations, Execution

**Day 30 — Certification-style Practice**
- Focus on: DataFrame API, Transformations, Spark behavior

---

## Day 1 — What is Apache Spark?

### The Big Idea

Spark is a **distributed compute engine**. You write Python (or SQL), and Spark runs it across a cluster of machines in parallel.

Think of it as: pandas, but the data is split across 10 (or 1000) machines and processed simultaneously.

---

### Core Concepts

#### 1. Driver vs Executors

```
Your Notebook (Driver)
        │
        ▼
   Spark Context
        │
   ┌────┴────┐
   ▼         ▼
Executor1  Executor2  ...  ExecutorN
(Worker)   (Worker)        (Worker)
```

- **Driver:** your code, your notebook — orchestrates the job, holds the plan
- **Executors:** worker nodes that actually process data — each handles a partition
- **SparkContext / SparkSession:** the entry point that connects driver to the cluster

In Databricks, `spark` is already available — no setup needed.

---

#### 2. Lazy Evaluation

Spark does **nothing** until you force it to.

- `select()`, `filter()`, `withColumn()` → **transformations** — just build a plan, no data moves
- `show()`, `count()`, `write()` → **actions** — trigger actual execution

Why? So Spark can optimize the full plan before running anything.

---

#### 3. RDD → DataFrame → Dataset

| Layer | What it is | Used today? |
|---|---|---|
| RDD | Raw distributed collection, low-level | Rarely |
| DataFrame | Distributed table with schema (like pandas) | Yes — this is your main tool |
| Dataset | Typed DataFrame (Scala/Java only) | Not in Python |

You will almost always work with **DataFrames** and **Spark SQL**.

In [ ]:
# Day 1 — Hands-on: your first Spark operations
# Run this in Databricks — spark is already available

# Check Spark version
print(spark.version)

In [ ]:
# Create a small DataFrame from scratch
data = [
    ("Alice", 34, "Engineering"),
    ("Bob",   28, "Marketing"),
    ("Carol", 41, "Engineering"),
    ("Dave",  25, "Sales"),
]
columns = ["name", "age", "department"]

df = spark.createDataFrame(data, columns)
df.show()

In [ ]:
# Inspect schema — Spark inferred types automatically
df.printSchema()

In [ ]:
# Lazy evaluation demo
# This builds a plan — nothing runs yet
filtered = df.filter(df.age > 30).select("name", "department")

# THIS triggers execution
filtered.show()

In [ ]:
# See the execution plan Spark built
filtered.explain()

---

## Day 2 — Lazy Evaluation Deep Dive

### Transformations vs Actions

| Transformations (lazy) | Actions (trigger execution) |
|---|---|
| `select()` | `show()` |
| `filter()` / `where()` | `count()` |
| `withColumn()` | `collect()` |
| `groupBy()` | `write()` |
| `join()` | `toPandas()` |
| `orderBy()` | `first()` / `take()` |

Chaining transformations is free — Spark just builds a DAG (Directed Acyclic Graph) of operations. The moment you call an action, Spark optimizes and executes the whole chain.

### Why this matters:
- You can build complex multi-step pipelines without wasting compute on intermediate results
- Spark's **Catalyst optimizer** rewrites your plan to be more efficient before running it
- `explain()` lets you see what Spark actually plans to do

In [ ]:
from pyspark.sql import functions as F

# Build a multi-step plan — nothing executes yet
result = (
    df
    .filter(F.col("age") > 25)
    .withColumn("age_group", F.when(F.col("age") >= 35, "Senior").otherwise("Junior"))
    .groupBy("department", "age_group")
    .agg(F.count("*").alias("headcount"))
    .orderBy("department")
)

# Action — now Spark runs the whole thing
result.show()

In [ ]:
# count() is an action — triggers execution and returns a Python int
n = df.count()
print(f"Total rows: {n}")

---

## ✅ Days 1–2 Checklist

- [ ] `spark.version` printed successfully
- [ ] Created a DataFrame with `spark.createDataFrame()`
- [ ] Used `show()`, `printSchema()`, `explain()`
- [ ] Understood: transformations are lazy, actions trigger execution
- [ ] Chained multiple transformations and ran as one job

**Next:** Day 3 — Spark Runtime Architecture (Jobs, Stages, Tasks)